In [1]:
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, Iterator, Optional
import re
import pandas as pd


CAP_TO_FF = {
    "FF": 1.0,
    "PF": 1e3,
    "NF": 1e6,
}

RES_TO_OHM = {
    "OHM": 1.0,
    "KOHM": 1e3,
}


@dataclass
class SpefUnits:
    c_unit_value: float = 1.0
    c_unit_name: str = "PF"
    r_unit_value: float = 1.0
    r_unit_name: str = "OHM"

    @property
    def cap_to_ff(self) -> float:
        return self.c_unit_value * CAP_TO_FF[self.c_unit_name.upper()]

    @property
    def res_to_ohm(self) -> float:
        return self.r_unit_value * RES_TO_OHM[self.r_unit_name.upper()]


def resolve_name(token: str, name_map: Dict[str, str]) -> str:
    """
    Resolves SPEF aliases:
      *5250 -> real net name if present in *NAME_MAP
      *5250:14 -> real_net_name:14
    """
    if not token.startswith("*"):
        return token

    if ":" in token:
        base, suffix = token.split(":", 1)
        return f"{name_map.get(base, base)}:{suffix}"

    return name_map.get(token, token)


def parse_spef_header_and_namemap(spef_path: Path) -> tuple[SpefUnits, Dict[str, str]]:
    units = SpefUnits()
    name_map: Dict[str, str] = {}

    in_name_map = False

    with spef_path.open("r", errors="ignore") as f:
        for line in f:
            line = line.strip()

            if not line:
                continue

            if line.startswith("*C_UNIT"):
                # Example: *C_UNIT 1 PF
                _, value, unit = line.split()[:3]
                units.c_unit_value = float(value)
                units.c_unit_name = unit.upper()

            elif line.startswith("*R_UNIT"):
                # Example: *R_UNIT 1 OHM
                _, value, unit = line.split()[:3]
                units.r_unit_value = float(value)
                units.r_unit_name = unit.upper()

            elif line.startswith("*NAME_MAP"):
                in_name_map = True
                continue

            elif line.startswith("*D_NET"):
                break

            elif in_name_map:
                # Example: *332 _00331_
                parts = line.split(maxsplit=1)
                if len(parts) == 2 and parts[0].startswith("*"):
                    name_map[parts[0]] = parts[1]

    return units, name_map


def parse_spef_nets(spef_path: Path) -> Iterator[dict]:
    units, name_map = parse_spef_header_and_namemap(spef_path)

    current = None
    section: Optional[str] = None

    with spef_path.open("r", errors="ignore") as f:
        for line in f:
            line = line.strip()

            if not line:
                continue

            if line.startswith("*D_NET"):
                # Flush previous net if something went wrong before *END
                if current is not None:
                    yield finalize_net(current, units)

                # Example: *D_NET *5250 6.94318e-05
                parts = line.split()
                if len(parts) < 3:
                    current = None
                    continue

                net_token = parts[1]
                total_cap_raw = float(parts[2])

                current = {
                    "net_name_spef": net_token,
                    "net_name": resolve_name(net_token, name_map),
                    "cap_total_raw": total_cap_raw,
                    "ground_cap_raw": 0.0,
                    "coupling_cap_raw": 0.0,
                    "res_total_raw": 0.0,
                    "num_cap_entries": 0,
                    "num_ground_cap_entries": 0,
                    "num_coupling_cap_entries": 0,
                    "num_res_entries": 0,
                }
                section = None
                continue

            if current is None:
                continue

            if line.startswith("*CONN"):
                section = "CONN"
                continue

            if line.startswith("*CAP"):
                section = "CAP"
                continue

            if line.startswith("*RES"):
                section = "RES"
                continue

            if line.startswith("*END"):
                yield finalize_net(current, units)
                current = None
                section = None
                continue

            if section == "CAP":
                parse_cap_line(line, current)

            elif section == "RES":
                parse_res_line(line, current)

    if current is not None:
        yield finalize_net(current, units)


def parse_cap_line(line: str, current: dict) -> None:
    """
    SPEF CAP line variants:
      1 node cap
      2 node1 node2 cap
    Two-node form is coupling capacitance.
    """
    parts = line.split()
    if len(parts) == 3:
        # ground capacitance
        _, node, cap = parts
        current["ground_cap_raw"] += float(cap)
        current["num_cap_entries"] += 1
        current["num_ground_cap_entries"] += 1

    elif len(parts) == 4:
        # coupling capacitance
        _, node1, node2, cap = parts
        current["coupling_cap_raw"] += float(cap)
        current["num_cap_entries"] += 1
        current["num_coupling_cap_entries"] += 1


def parse_res_line(line: str, current: dict) -> None:
    """
    SPEF RES line:
      1 node1 node2 resistance
    """
    parts = line.split()
    if len(parts) == 4:
        _, node1, node2, res = parts
        current["res_total_raw"] += float(res)
        current["num_res_entries"] += 1


def finalize_net(current: dict, units: SpefUnits) -> dict:
    result = dict(current)

    result["cap_total_ff"] = result["cap_total_raw"] * units.cap_to_ff
    result["ground_cap_ff"] = result["ground_cap_raw"] * units.cap_to_ff
    result["coupling_cap_ff"] = result["coupling_cap_raw"] * units.cap_to_ff
    result["res_total_ohm"] = result["res_total_raw"] * units.res_to_ohm

    result["spef_c_unit"] = f"{units.c_unit_value:g} {units.c_unit_name}"
    result["spef_r_unit"] = f"{units.r_unit_value:g} {units.r_unit_name}"

    return result


def extract_spef_to_dataframe(spef_path: str | Path) -> pd.DataFrame:
    spef_path = Path(spef_path)
    rows = list(parse_spef_nets(spef_path))
    return pd.DataFrame(rows)

In [2]:
if __name__ == "__main__":
    spef = Path("/home/nvgel/phd/dataset/asap7/ac97_top/CLK_400_IO_0.00_CU_20_AR_0.5_HW_4.5_HS_4.5_HP_32_VW_4.5_VS_4.5_VP_32/route/spef/spef.spef")
    df = extract_spef_to_dataframe(spef)

    print(df.head())
    print(df[["cap_total_ff", "ground_cap_ff", "coupling_cap_ff"]].describe())

    df.to_parquet("net_capacitance_labels.parquet", index=False)

  net_name_spef net_name  cap_total_raw  ground_cap_raw  coupling_cap_raw  \
0          *332  _00331_       0.000306        0.000143          0.000163   
1          *333  _00332_       0.000176        0.000086          0.000090   
2          *334  _00333_       0.000179        0.000107          0.000072   
3          *335  _00334_       0.001024        0.000475          0.000548   
4          *336  _00335_       0.000251        0.000185          0.000066   

   res_total_raw  num_cap_entries  num_ground_cap_entries  \
0      52.140302               25                      11   
1      30.375362               21                       7   
2      51.524232               15                       7   
3     192.169688               46                      15   
4      53.489766               20                       9   

   num_coupling_cap_entries  num_res_entries  cap_total_ff  ground_cap_ff  \
0                        14               10      0.305986       0.142810   
1               

In [3]:
df['cap_total_raw'].max()

np.float64(0.0228568)

In [5]:
df = pd.read_parquet("/home/nvgel/phd/OpenROAD-flow-Extract/data.parquet")

ArrowInvalid: Could not open Parquet input source '<Buffer>': Parquet magic bytes not found in footer. Either the file is corrupted or this is not a parquet file.